# Abdomen CT — nnUNet v2 Pipeline
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle: `ramazan2020/abdomen-acikveri`
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır — veri indirme gerekmez

**Colab'da çalıştırma:**
1. `Runtime → Change runtime type → GPU`
2. Kaggle Secrets'a `KAGGLE_USERNAME` + `KAGGLE_KEY` ekle
3. Google Drive bağlantısı kur (veri Drive'dan okunur)

---

| Adım | Süre (T. yakl.) |
|------|-----------------|
| DICOM → NIfTI | 20–40 dk |
| Plan + Preprocess | 15–30 dk |
| Eğitim 3d_fullres | 8–12 saat |
| Inference + Değerlendirme | 20–30 dk |

> **Eğitim uzun:** Kaggle 9 saat session sınırı vardır. Checkpoint otomatik kaydedilir;  
> session bitince `CONTINUE_TRAINING = True` yapıp yeniden başlatın.

---
## 0. Ortam Tespiti ve GPU Kontrolü

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

print(f"Ortam : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Yerel'}")

import torch
if not torch.cuda.is_available():
    _msg = (
        "GPU bulunamadı!\n"
        "Kaggle: Settings → Accelerator → GPU\n"
        "Colab : Runtime → Change runtime type → GPU"
    )
    raise RuntimeError(_msg)

print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch CUDA: {torch.version.cuda}")

r = subprocess.run(['df', '-h', '.'], capture_output=True, text=True)
print(f"\nDisk:\n{r.stdout}")

---
## 1. Ortam Kurulumu

- **Kaggle**: Sadece kütüphane kurulumu
- **Colab**: Kaggle API kimlik doğrulama + Google Drive bağlantısı

In [ ]:
if IS_COLAB:
    # ── Kaggle API ─────────────────────────────────────────────────────────
    # Colab sidebar'dan 🔑 simgesine tıklayın ve KAGGLE_USERNAME, KAGGLE_KEY ekleyin
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print("Kaggle kimlik bilgileri Colab Secrets'tan yüklendi")
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print("kaggle.json dosyasını seçin:")
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print("kaggle.json yüklendi")

    # ── Google Drive ────────────────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive bağlandı")

else:
    print("Kaggle ortamı — API kurulumu ve Drive atlandı")

---
## 2. Kütüphane Kurulumu

In [ ]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'nnunetv2', 'SimpleITK', 'pydicom', 'scipy', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()

import nnunetv2, SimpleITK, pydicom, scipy, tqdm
print(f"nnunetv2  : {nnunetv2.__version__}")
print(f"SimpleITK : {SimpleITK.__version__}")
print(f"pydicom   : {pydicom.__version__}")

import shutil, sysconfig
def find_nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path('scripts'), str(Path(sys.executable).parent),
              '/usr/local/bin', '/opt/conda/bin']:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

for cmd in ['nnUNetv2_plan_and_preprocess', 'nnUNetv2_train', 'nnUNetv2_predict']:
    p = find_nnunet_cmd(cmd)
    print(f"  {cmd}: {Path(p).exists()}")

---
## 3. Konfigürasyon

**Kaggle**: Dataset slug'ını `KAGGLE_DATASET_SLUG` ile belirtin (input dizinindeki klasör adı).  
**Colab**: Dataset Kaggle'dan indirilir, Drive'a preprocessed kaydedilir.

In [ ]:
import os, sys, json, shutil, time, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
# Kaggle: "Datasets" sekmesinden eklenen dataset'in klasör adı (/kaggle/input/ altında)
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'         # /kaggle/input/{slug}/
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'  # Colab indirme için

FOLD         = 0    # Hangi fold
DATASET_ID   = 100  # nnUNet dataset numarası
DATASET_NAME = 'Abdomen'
# ──────────────────────────────────────────────────────────────────────────

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis',         # 0 → label 1
    'kidney_ureter_stone',         # 1 → label 2
    'acute_pancreatitis',          # 2 → label 3
    'aortic_aneurysm_dissection',  # 3 → label 4
    'acute_appendicitis',          # 4 → label 5
    'acute_diverticulitis',        # 5 → label 6
]

if IS_KAGGLE:
    # ── Kaggle: veri zaten mount edilmiş ──────────────────────────────────
    DATA_DIR    = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR    = Path('/kaggle/working')
    NIFTI_DIR   = WORK_DIR / 'nifti'
    NNUNET_RAW            = WORK_DIR / 'nnunet_raw'
    NNUNET_PREPROCESSED_P = WORK_DIR / 'nnunet_preprocessed'  # P=persistent working
    NNUNET_RESULTS_P      = WORK_DIR / 'nnunet_results'

    # Önceki session'dan checkpoint geldi mi?
    _checkpoint_dataset = Path('/kaggle/input/nnunet-checkpoint')
    CHECKPOINT_INPUT = _checkpoint_dataset if _checkpoint_dataset.exists() else None

elif IS_COLAB:
    # ── Colab: Drive kalıcı, lokal geçici ────────────────────────────────
    DATA_DIR    = Path('/content/kaggle_data')
    DRIVE_BASE  = Path('/content/drive/MyDrive/Abdomen')
    NIFTI_DIR   = Path('/content/nifti')              # lokal (hızlı)
    NNUNET_RAW            = DRIVE_BASE / 'nnunet_raw' # Drive (kalıcı)
    NNUNET_PREPROCESSED_D = DRIVE_BASE / 'nnunet_preprocessed'   # Drive
    NNUNET_PREPROCESSED_L = Path('/content/nnunet_preprocessed') # lokal (hızlı)
    NNUNET_RESULTS_D      = DRIVE_BASE / 'nnunet_results'        # Drive
    NNUNET_RESULTS_L      = Path('/content/nnunet_results')      # lokal
    NNUNET_PREPROCESSED_P = NNUNET_PREPROCESSED_L
    NNUNET_RESULTS_P      = NNUNET_RESULTS_L
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

EGITIM_DIR = DATA_DIR / 'Egitim Verisi'
SPLIT_DIR  = DATA_DIR / 'outputs' / 'splits'
DATASET_DIR = NNUNET_RAW / f'Dataset{DATASET_ID}_{DATASET_NAME}'

for p in [
    NIFTI_DIR, NNUNET_RAW, NNUNET_PREPROCESSED_P, NNUNET_RESULTS_P,
    DATASET_DIR / 'imagesTr', DATASET_DIR / 'labelsTr',
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
os.environ['nnUNet_results']      = str(NNUNET_RESULTS_P)
os.environ['OMP_NUM_THREADS']     = '1'

print('Konfigürasyon:')
print(f'  Ortam          : {"Kaggle" if IS_KAGGLE else "Colab"}')
print(f'  DATA_DIR       : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'  EGITIM_DIR     : {EGITIM_DIR}  (var={EGITIM_DIR.exists()})')
print(f'  SPLIT_DIR      : {SPLIT_DIR}  (var={SPLIT_DIR.exists()})')
print(f'  NIFTI_DIR      : {NIFTI_DIR}')
print(f'  nnUNet raw     : {NNUNET_RAW}')
print(f'  nnUNet preproc : {NNUNET_PREPROCESSED_P}')
print(f'  nnUNet results : {NNUNET_RESULTS_P}')

if IS_KAGGLE and not EGITIM_DIR.exists():
    raise FileNotFoundError(
        f'Dataset bulunamadı: {DATA_DIR}\n'
        f'Kaggle sağ panelden Datasets → "{KAGGLE_DATASET_SLUG}" ekleyin.'
    )

---
## 4. Veri İndirme (Yalnızca Colab)

**Kaggle'da bu hücre otomatik atlanır.**

In [ ]:
if IS_KAGGLE:
    print("Kaggle: Dataset zaten mount edilmiş, indirme atlandı")
    print(f"  Egitim Verisi: {len(list(EGITIM_DIR.iterdir()))} vaka")
else:
    # Colab: Kaggle'dan indir
    _already = EGITIM_DIR.exists() and any(EGITIM_DIR.iterdir())
    if _already:
        print(f"Veri zaten mevcut: {len(list(EGITIM_DIR.iterdir()))} vaka")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}")
        print("Bu işlem 30-90 dakika sürebilir...")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-1000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({time.time()-t0:.0f}s)")

    # Colab: preprocessed Drive'da varsa lokal kopyasını al
    _drive_pre = NNUNET_PREPROCESSED_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _local_pre = NNUNET_PREPROCESSED_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _drive_pre.exists() and not _local_pre.exists():
        print('Preprocessed Drive→Local kopyalanıyor...')
        t0 = time.time()
        shutil.copytree(str(_drive_pre), str(_local_pre))
        print(f'Kopyalandı ({time.time()-t0:.0f}s)')
        os.environ['nnUNet_results'] = str(NNUNET_RESULTS_L)

assert EGITIM_DIR.exists(), f'EGITIM_DIR bulunamadı: {EGITIM_DIR}'
assert SPLIT_DIR.exists(),  f'SPLIT_DIR bulunamadı: {SPLIT_DIR}'
print("Veri doğrulandı")

---
## 5. Yardımcı Fonksiyonlar

In [ ]:
import pydicom

def load_fold(fold: int, split: str) -> List[str]:
    p = SPLIT_DIR / f'fold{fold}_{split}.csv'
    return pd.read_csv(p)['Case Number'].astype(str).tolist()

def raw_case_id(case: str) -> int:
    """'T_20001' → 20001"""
    return int(str(case).split('_', 1)[-1])

def build_z_map(case: str) -> dict:
    """DICOM z-pozisyonuna göre image_id → z_index haritası."""
    raw = str(case).split('_', 1)[-1]
    case_dir = EGITIM_DIR / raw
    dcm_files = sorted(
        [p for p in case_dir.glob('*.dcm') if not p.stem.startswith('.')],
        key=lambda p: int(p.stem)
    )
    positions = []
    for p in dcm_files:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        ipp = getattr(ds, 'ImagePositionPatient', None)
        zpos = float(ipp[2]) if ipp else float(p.stem)
        positions.append((int(p.stem), zpos))
    positions.sort(key=lambda x: x[1])
    return {img_id: idx for idx, (img_id, _) in enumerate(positions)}

def _iou2d(a, b) -> float:
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    iw = max(0.0, min(ax2,bx2) - max(ax1,bx1))
    ih = max(0.0, min(ay2,by2) - max(ay1,by1))
    inter = iw * ih
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    return inter / max(ua + ub - inter, 1e-6)

def lift_2d_to_3d(manifest: pd.DataFrame, case: str,
                   delta_z: int = 2, iou_th: float = 0.3) -> list:
    """Manifest'teki 2D annotasyonları 3D bounding box'lara dönüştürür."""
    z_map = build_z_map(case)
    sub = manifest[manifest['case'] == case].copy()
    sub = sub[sub['bboxes'].fillna('').astype(str) != '']
    items = []
    for _, row in sub.iterrows():
        z = z_map.get(int(row['image_id']))
        if z is None: continue
        for bb_str in str(row['bboxes']).split('|'):
            parts = bb_str.strip().split(',')
            if len(parts) < 5: continue
            try:
                sid, x1, y1, x2, y2 = (int(float(v)) for v in parts[:5])
            except (ValueError, TypeError):
                continue
            if x2 > x1 and y2 > y1:
                items.append((sid, x1, y1, x2, y2, z))
    boxes_3d = []
    for sid in set(it[0] for it in items):
        cls_items = sorted([it for it in items if it[0] == sid], key=lambda x: x[5])
        used: set = set()
        for i, it in enumerate(cls_items):
            if i in used: continue
            grp = [it]; used.add(i)
            for j in range(i+1, len(cls_items)):
                if j in used: continue
                last = grp[-1]
                if cls_items[j][5] - last[5] <= delta_z:
                    if _iou2d(last[1:5], cls_items[j][1:5]) >= iou_th:
                        grp.append(cls_items[j]); used.add(j)
                else: break
            boxes_3d.append({
                'class'   : sid,
                'x1': min(g[1] for g in grp), 'y1': min(g[2] for g in grp),
                'z1': min(g[5] for g in grp),
                'x2': max(g[3] for g in grp), 'y2': max(g[4] for g in grp),
                'z2': max(g[5] for g in grp), 'n_slices': len(grp),
            })
    return boxes_3d

# Fold yükle
train_cases = load_fold(FOLD, 'train')
val_cases   = load_fold(FOLD, 'val')
all_cases   = sorted(set(train_cases + val_cases))
print(f'Fold {FOLD}: {len(train_cases)} train + {len(val_cases)} val = {len(all_cases)} toplam')
print('Yardımcı fonksiyonlar yüklendi')

In [ ]:
# ─── Değerlendirme Fonksiyonları ──────────────────────────────────────────
def _iou_xyxy(a, b) -> float:
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    iw=max(0.,min(ax2,bx2)-max(ax1,bx1)); ih=max(0.,min(ay2,by2)-max(ay1,by1))
    inter=iw*ih; ua=max(0.,(ax2-ax1)*(ay2-ay1)); ub=max(0.,(bx2-bx1)*(by2-by1))
    return inter/max(ua+ub-inter,1e-6)

def _match(pred_df, gt_df, iou_th):
    tp=fp=0; matched=set()
    if 'score' in pred_df.columns: pred_df=pred_df.sort_values('score',ascending=False)
    gt_grp=gt_df.groupby(['case','image_id'])
    for _,pr in pred_df.iterrows():
        key=(pr['case'],pr['image_id'])
        if key not in gt_grp.groups: fp+=1; continue
        cand=gt_grp.get_group(key); best_iou,best_idx=0.,-1
        for idx,g in cand.iterrows():
            if (pr['case'],pr['image_id'],idx) in matched: continue
            iou=_iou_xyxy((pr['x1'],pr['y1'],pr['x2'],pr['y2']),(g['x1'],g['y1'],g['x2'],g['y2']))
            if iou>best_iou: best_iou,best_idx=iou,idx
        if best_iou>=iou_th: tp+=1; matched.add((pr['case'],pr['image_id'],best_idx))
        else: fp+=1
    return tp,fp,len(gt_df)-len(matched)

def f1_at_iou(pred_df, gt_df, iou_th):
    classes=sorted(set(pred_df['class']).union(set(gt_df['class'])))
    per_cls,tp_t,fp_t,fn_t={},0,0,0
    for cls in classes:
        tp,fp,fn=_match(pred_df[pred_df['class']==cls],gt_df[gt_df['class']==cls],iou_th)
        tp_t+=tp; fp_t+=fp; fn_t+=fn
        prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
        per_cls[cls]={'precision':prec,'recall':rec,'f1':2*prec*rec/max(prec+rec,1e-9),'tp':tp,'fp':fp,'fn':fn}
    macro=float(np.mean([v['f1'] for v in per_cls.values()])) if per_cls else 0.
    mp=tp_t/max(tp_t+fp_t,1); mr=tp_t/max(tp_t+fn_t,1)
    return {'per_class':per_cls,'macro_f1':macro,'micro_f1':2*mp*mr/max(mp+mr,1e-9)}

def top5_f1_mean(pred_df, gt_df, ths=(0.1,0.2,0.3,0.4,0.5)):
    f1s=[(th,f1_at_iou(pred_df,gt_df,th)['macro_f1']) for th in ths]
    top5=sorted(f1s,key=lambda x:x[1],reverse=True)[:5]
    return {'per_threshold':f1s,'top5':top5,'top5_mean_f1':float(np.mean([v for _,v in top5]))}

print('Değerlendirme fonksiyonları yüklendi')

---
## 6. DICOM → NIfTI Dönüşümü

**Kaggle**: Her session'da yeniden çalışır (working disk session ömürlüdür).  
Paralel dönüşüm — mevcut dosyalar atlanır.

In [ ]:
import SimpleITK as sitk
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

def dicom_to_nifti(case: str) -> str:
    raw = raw_case_id(case)
    out = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
    if out.exists(): return 'skip'
    case_dir = EGITIM_DIR / str(raw)
    if not case_dir.exists(): return f'missing:{case}'
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names: return f'no_dcm:{case}'
    reader.SetFileNames(names)
    try:
        sitk.WriteImage(reader.Execute(), str(out))
        return 'ok'
    except Exception as e:
        return f'err:{e}'

print(f'DICOM→NIfTI: {len(all_cases)} vaka  |  Kaynak: {EGITIM_DIR}')
_nii_existing = len(list(NIFTI_DIR.glob('*.nii.gz')))
print(f'Mevcut NIfTI: {_nii_existing}')

with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(tqdm(ex.map(dicom_to_nifti, all_cases),
                        total=len(all_cases), desc='DICOM→NIfTI'))

ok   = sum(1 for r in results if r == 'ok')
skip = sum(1 for r in results if r == 'skip')
errs = [r for r in results if r not in ('ok', 'skip')]
print(f'{ok} yeni  |  {skip} atlandı  |  {len(errs)} hata')
if errs: print('Hatalar:', errs[:5])
print(f'Toplam NIfTI: {len(list(NIFTI_DIR.glob("*.nii.gz")))}')

---
## 7. nnUNet Dataset Formatı Hazırlama

- `imagesTr/` → CT hacimler (symlink)
- `labelsTr/` → Semantic maske (0=arka plan, 1-6=patoloji sınıfları)
- `dataset.json` → nnUNet meta

In [ ]:
manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
print(f'Manifest: {len(manifest)} satır  |  Sütunlar: {list(manifest.columns)}')

def prep_case_nnunet(case: str) -> str:
    raw = raw_case_id(case)
    src_nii = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
    if not src_nii.exists(): return f'missing:{case}'

    dst_img = DATASET_DIR / 'imagesTr' / f'case_{raw:05d}_0000.nii.gz'
    if not dst_img.exists():
        try: os.symlink(src_nii, dst_img)
        except (OSError, NotImplementedError): shutil.copy2(str(src_nii), str(dst_img))

    dst_lbl = DATASET_DIR / 'labelsTr' / f'case_{raw:05d}.nii.gz'
    if dst_lbl.exists(): return 'skip'

    boxes = lift_2d_to_3d(manifest, case)
    img_itk = sitk.ReadImage(str(src_nii))
    shape = sitk.GetArrayFromImage(img_itk).shape  # (Z, Y, X)
    mask = np.zeros(shape, dtype=np.uint8)
    for b in boxes:
        label = int(b['class']) + 1
        mask[int(b['z1']):min(int(b['z2'])+1,shape[0]),
             int(b['y1']):min(int(b['y2'])+1,shape[1]),
             int(b['x1']):min(int(b['x2'])+1,shape[2])] = label
    m = sitk.GetImageFromArray(mask)
    m.CopyInformation(img_itk)
    sitk.WriteImage(m, str(dst_lbl))
    return 'ok'

print(f'Dataset hazırlanıyor: {len(all_cases)} vaka...')
results = [prep_case_nnunet(c) for c in tqdm(all_cases, desc='Dataset prep')]

ok=results.count('ok'); skip=results.count('skip')
miss=[r for r in results if 'missing' in r]
print(f'{ok} yeni  |  {skip} atlandı  |  {len(miss)} eksik NIfTI')

# dataset.json
dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {'background': 0, **{n: i+1 for i,n in enumerate(SUPER_CLASSES)}},
    'numTraining': len(all_cases),
    'file_ending': '.nii.gz',
}
(DATASET_DIR/'dataset.json').write_text(json.dumps(dataset_json, indent=2))

n_img = len(list((DATASET_DIR/'imagesTr').glob('*.nii.gz')))
n_lbl = len(list((DATASET_DIR/'labelsTr').glob('*.nii.gz')))
print(f'imagesTr: {n_img}  |  labelsTr: {n_lbl}')
if n_img == 0: raise RuntimeError('imagesTr boş!')

---
## 8. nnUNet Plan + Preprocess

Fingerprint çıkarımı, plan oluşturma ve veriyi nnUNet formatına dönüştürme.  
**Önceki session'da tamamlandıysa bu hücre atlanır.**

In [ ]:
_plans_file = (NNUNET_PREPROCESSED_P
               / f'Dataset{DATASET_ID}_{DATASET_NAME}'
               / 'nnUNetPlans.json')

if _plans_file.exists():
    print(f'Plans dosyası mevcut, preprocess atlandı: {_plans_file}')
else:
    _cmd = find_nnunet_cmd('nnUNetv2_plan_and_preprocess')
    print(f'Plan + Preprocess başlatılıyor...')
    print(f'  Dataset: {DATASET_ID}, Config: 3d_fullres')
    print('  ~15–30 dakika sürebilir...')

    _proc = subprocess.Popen(
        [_cmd, '-d', str(DATASET_ID), '-c', '3d_fullres',
         '--verify_dataset_integrity'],
        env={**os.environ},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for line in _proc.stdout:
        print(line, end='', flush=True)
    _proc.wait()
    if _proc.returncode != 0:
        raise RuntimeError(f'plan_and_preprocess başarısız! Kod: {_proc.returncode}')
    print('\nPlan + Preprocess tamamlandı!')

    # Colab: preprocessed'i Drive'a yedekle
    if IS_COLAB:
        _src = NNUNET_PREPROCESSED_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
        _dst = NNUNET_PREPROCESSED_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
        if _src.exists() and not _dst.exists():
            print('Preprocessed Drive\'a yedekleniyor...')
            shutil.copytree(str(_src), str(_dst))
            print('Yedeklendi')

In [ ]:
# Özel fold split yaz (nnUNet auto-split'ini geçersiz kıl)
_splits_dir = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
_splits_dir.mkdir(parents=True, exist_ok=True)

_splits = [{
    'train': [f'case_{raw_case_id(c):05d}' for c in train_cases],
    'val'  : [f'case_{raw_case_id(c):05d}' for c in val_cases],
}]
(_splits_dir / 'splits_final.json').write_text(json.dumps(_splits, indent=2))
print(f'splits_final.json: {len(train_cases)} train, {len(val_cases)} val')

---
## 9. Eğitim

**Kaggle 9 saat session sınırı**: Eğitim otomatik checkpoint kaydeder.  
Session bittiğinde:
1. `/kaggle/working/nnunet_results/` klasörünü Kaggle output olarak kaydedin
2. Yeni session'da bu output'u dataset olarak ekleyin
3. `CONTINUE_TRAINING = True` yapıp çalıştırın

In [ ]:
CONTINUE_TRAINING = False  # Önceki session'dan devam etmek için True yapın

if IS_KAGGLE and CONTINUE_TRAINING:
    # Önceki session checkpoint'ini working dizinine kopyala
    _ckpt_src = Path('/kaggle/input/nnunet-checkpoint')
    if _ckpt_src.exists():
        print(f'Checkpoint yükleniyor: {_ckpt_src}')
        _ckpt_dst = NNUNET_RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
        if not _ckpt_dst.exists():
            t0 = time.time()
            shutil.copytree(str(_ckpt_src / f'Dataset{DATASET_ID}_{DATASET_NAME}'),
                            str(_ckpt_dst))
            print(f'Checkpoint kopyalandı ({time.time()-t0:.0f}s)')
        else:
            print('Checkpoint zaten mevcut')
    else:
        print(f'Checkpoint dataset bulunamadı: {_ckpt_src}')
        print('Sağ panelden Datasets → nnunet-checkpoint ekleyin')
elif IS_COLAB and CONTINUE_TRAINING:
    _ckpt_drive = NNUNET_RESULTS_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _ckpt_local = NNUNET_RESULTS_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _ckpt_drive.exists() and not _ckpt_local.exists():
        print('Checkpoint Drive\'dan yükleniyor...')
        shutil.copytree(str(_ckpt_drive), str(_ckpt_local))
        print('Yüklendi')

print(f'CONTINUE_TRAINING: {CONTINUE_TRAINING}')

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU gerekli!'

_cmd = find_nnunet_cmd('nnUNetv2_train')
print(f'=== Eğitim Başlıyor ===')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Dataset : {DATASET_ID}, Config: 3d_fullres, Fold: {FOLD}')
print(f'Devam   : {CONTINUE_TRAINING}')
print(f'Results : {os.environ["nnUNet_results"]}')
print('─' * 50)

_args = [_cmd, str(DATASET_ID), '3d_fullres', str(FOLD), '--npz']
if CONTINUE_TRAINING:
    _args.append('--c')

_proc = subprocess.Popen(
    _args, env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

print('─' * 50)
if _proc.returncode == 0:
    print('Eğitim tamamlandı!')
else:
    print(f'Eğitim kodu: {_proc.returncode} (session kesildi veya hata)')

# Colab: modeli Drive'a yedekle
if IS_COLAB:
    _src = NNUNET_RESULTS_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst = NNUNET_RESULTS_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _src.exists():
        if _dst.exists(): shutil.rmtree(str(_dst))
        shutil.copytree(str(_src), str(_dst))
        print(f'Model Drive\'a yedeklendi: {_dst}')

---
## 10. Tahmin (Inference) — Validasyon Seti

In [ ]:
VAL_INPUT_DIR  = NNUNET_RESULTS_P.parent / 'val_input'
VAL_OUTPUT_DIR = (NNUNET_RESULTS_P
                  / f'Dataset{DATASET_ID}_{DATASET_NAME}'
                  / 'nnUNetTrainer__nnUNetPlans__3d_fullres'
                  / f'fold_{FOLD}'
                  / 'val_predictions')
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_linked = 0
for c in val_cases:
    raw = raw_case_id(c)
    src = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
    dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
    if src.exists() and not dst.exists():
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
        _linked += 1

print(f'Val input: {len(list(VAL_INPUT_DIR.glob("*.nii.gz")))} dosya')
print(f'Tahmin çıktı: {VAL_OUTPUT_DIR}')

_cmd = find_nnunet_cmd('nnUNetv2_predict')
print('Tahmin başlatılıyor...')

_proc = subprocess.Popen(
    [_cmd,
     '-i', str(VAL_INPUT_DIR),
     '-o', str(VAL_OUTPUT_DIR),
     '-d', str(DATASET_ID),
     '-c', '3d_fullres',
     '-f', str(FOLD),
     '--save_probabilities'],
    env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

_preds = list(VAL_OUTPUT_DIR.glob('*.nii.gz'))
print(f'\nTahmin tamamlandı: {len(_preds)} NIfTI mask')

---
## 11. Değerlendirme — F1 Skoru

In [ ]:
from scipy import ndimage

def seg_to_pred_df(pred_dir: Path) -> pd.DataFrame:
    """NIfTI segmentasyon maskeleri → prediction DataFrame (2D projeksiyon)."""
    rows = []
    for nii_path in sorted(pred_dir.glob('*.nii.gz')):
        try: raw = int(nii_path.stem.split('_')[1])
        except: continue
        mask = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_path)))  # [Z,Y,X]
        prob_path = nii_path.with_suffix('').with_suffix('.npz')
        probs = np.load(str(prob_path))['probabilities'] if prob_path.exists() else None
        for label_id in range(1, len(SUPER_CLASSES)+1):
            binary = (mask == label_id).astype(np.uint8)
            if binary.sum() == 0: continue
            labeled, n_comp = ndimage.label(binary)
            total_vox = float(binary.sum())
            for comp_id in range(1, n_comp+1):
                cm = (labeled == comp_id)
                coords = np.where(cm)
                z1,z2 = int(coords[0].min()),int(coords[0].max())
                y1,y2 = int(coords[1].min()),int(coords[1].max())
                x1,x2 = int(coords[2].min()),int(coords[2].max())
                score = float(probs[label_id][cm].mean()) if probs is not None else float(cm.sum())/total_vox
                for z in range(z1, z2+1):
                    rows.append({'case':raw,'image_id':z,'class':label_id-1,'score':score,
                                 'x1':float(x1),'y1':float(y1),'x2':float(x2),'y2':float(y2)})
    return pd.DataFrame(rows)

pred_df = seg_to_pred_df(VAL_OUTPUT_DIR)
print(f'Prediction: {len(pred_df):,} satır, {pred_df["case"].nunique() if not pred_df.empty else 0} vaka')

# Ground-truth
gt_rows = []
for case in val_cases:
    raw = raw_case_id(case)
    for b in lift_2d_to_3d(manifest, case):
        for z in range(int(b['z1']), int(b['z2'])+1):
            gt_rows.append({'case':raw,'image_id':z,'class':int(b['class']),
                            'x1':float(b['x1']),'y1':float(b['y1']),
                            'x2':float(b['x2']),'y2':float(b['y2'])})
gt_df = pd.DataFrame(gt_rows)
print(f'GT        : {len(gt_df):,} satır, {gt_df["case"].nunique() if not gt_df.empty else 0} vaka')

In [ ]:
if pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin.')
else:
    top5   = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)

    print(f'=== nnUNet v2 — Fold {FOLD} Sonuçları ===')
    print(f'Top-5 Mean F1 (makale metriği) : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @ IoU=0.3             : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @ IoU=0.3             : {detail["micro_f1"]:.4f}')
    print()
    print('IoU eşiğine göre Macro F1:')
    for th, f1v in top5['per_threshold']:
        print(f'  {th:.1f}: {f1v:.4f}')
    print()
    print('Sınıf bazında @ IoU=0.3:')
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30}  '
                  f'P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}  '
                  f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')

---
## 12. Sonuçları Kaydet

**Kaggle**: Çıktılar `/kaggle/working/` altında otomatik kaydedilir.  
Notebook'u commit edin → "Save Version" → output olarak kullanılabilir hale gelir.  
**Colab**: Drive'a kopyalanır.

In [ ]:
OUTPUT_DIR = NNUNET_RESULTS_P.parent / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV'ler
if not pred_df.empty:
    pred_df.to_csv(OUTPUT_DIR / f'pred_fold{FOLD}.csv', index=False)
    gt_df.to_csv(OUTPUT_DIR   / f'gt_fold{FOLD}.csv',   index=False)

    summary = {
        'fold': FOLD,
        'val_cases': len(val_cases),
        'top5_mean_f1': top5['top5_mean_f1'],
        'macro_f1_03' : detail['macro_f1'],
        'micro_f1_03' : detail['micro_f1'],
        'per_threshold': {str(th): float(f1v) for th, f1v in top5['per_threshold']},
    }
    (OUTPUT_DIR / f'summary_fold{FOLD}.json').write_text(json.dumps(summary, indent=2))
    print(f'Çıktılar: {OUTPUT_DIR}')
    for f in sorted(OUTPUT_DIR.glob('*')):
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

# Colab: Drive'a kopyala
if IS_COLAB:
    _dst = DRIVE_BASE / 'output'
    if _dst.exists(): shutil.rmtree(str(_dst))
    shutil.copytree(str(OUTPUT_DIR), str(_dst))
    print(f'\nDrive\'a kopyalandı: {_dst}')

print('\nTamamlandı!')
if IS_KAGGLE:
    print('\nCheckpoint kaydetmek için:')
    print('  Save Version → sağ üst köşe → Save & Run All')
    print('  Çıktılar otomatik /kaggle/working/ altında saklanır')